# Intelligent Email Spam Filtering System Using Naive Bayes

This notebook uses the real **Spambase** dataset and improves Naive Bayes accuracy using stronger NB variants, binarization, and hyperparameter tuning.

## What makes this version more accurate?
- Compares multiple Naive Bayes variants instead of using only GaussianNB
- Uses a binarized feature space for sparse/noisy email signals
- Tunes threshold and smoothing hyperparameters with stratified cross-validation
- Evaluates only once on a held-out test set

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import BernoulliNB, ComplementNB, GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Binarizer, StandardScaler

In [ ]:
# Load the real spam dataset from OpenML.
# Spambase contains numeric features extracted from email text.
spambase = fetch_openml(name='spambase', version=1, as_frame=True)
df = spambase.frame.copy()

X = df.drop(columns=['class'])
y = df['class'].astype(int)

print('Dataset shape:', df.shape)
print('Class distribution:')
print(y.value_counts().rename(index={0: 'Ham', 1: 'Spam'}))

In [ ]:
# Stratified hold-out split keeps the spam/ham ratio stable.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print('Train size:', X_train.shape)
print('Test size :', X_test.shape)

In [ ]:
# Compare Naive Bayes variants on the training split using stratified CV.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

candidates = {
    'GaussianNB + StandardScaler': Pipeline([
        ('scaler', StandardScaler()),
        ('nb', GaussianNB(var_smoothing=1e-9)),
    ]),
    'BernoulliNB + Binarizer(0.10)': Pipeline([
        ('bin', Binarizer(threshold=0.10)),
        ('nb', BernoulliNB(alpha=1.0)),
    ]),
    'ComplementNB + Binarizer(0.05)': Pipeline([
        ('bin', Binarizer(threshold=0.05)),
        ('nb', ComplementNB(alpha=1.0, norm=False)),
    ]),
}

rows = []
for name, model in candidates.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1'},
        n_jobs=None,
    )
    rows.append({
        'model': name,
        'accuracy_mean': scores['test_accuracy'].mean(),
        'accuracy_std': scores['test_accuracy'].std(),
        'precision_mean': scores['test_precision'].mean(),
        'recall_mean': scores['test_recall'].mean(),
        'f1_mean': scores['test_f1'].mean(),
    })

benchmark = pd.DataFrame(rows).sort_values('accuracy_mean', ascending=False)
benchmark

In [ ]:
# Hyperparameter tuning for the best NB family: ComplementNB.
# Binarizer threshold is the main signal-shaping parameter for this dataset.
best_nb = Pipeline([
    ('bin', Binarizer()),
    ('nb', ComplementNB()),
])

param_grid = {
    'bin__threshold': [0.01, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20],
    'nb__alpha': [0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0],
    'nb__norm': [False, True],
}

grid = GridSearchCV(
    best_nb,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=None,
)
grid.fit(X_train, y_train)

print('Best CV accuracy:', round(grid.best_score_, 4))
print('Best parameters:')
print(grid.best_params_)

In [ ]:
# Final evaluation on the untouched test set.
final_model = grid.best_estimator_
y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print('Final test metrics')
print('Accuracy :', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall   :', round(recall, 4))
print('F1-score :', round(f1, 4))
print()
print('Confusion matrix [ [TN, FP], [FN, TP] ]')
print(cm)
print()
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Save the tuned model for reuse.
out_path = Path('spam_filter_model.joblib')
joblib.dump(
    {
        'model': final_model,
        'best_params': grid.best_params_,
        'cv_accuracy': grid.best_score_,
        'feature_columns': list(X.columns),
    },
    out_path,
)
print('Saved to:', out_path.resolve())

In [ ]:
# Example predictions from the test split.
sample = X_test.iloc[:5].copy()
proba = final_model.predict_proba(sample)[:, 1]
preds = final_model.predict(sample)

demo = pd.DataFrame({
    'predicted': ['Spam' if p == 1 else 'Ham' for p in preds],
    'actual': ['Spam' if v == 1 else 'Ham' for v in y_test.iloc[:5].tolist()],
    'spam_probability': np.round(proba, 4),
})
demo